# Utilizing Rule-Based Semantic Extraction and Text Parsing to Automate Scoring Cards in *Magic: The Gathering*
### IMT 575 - Spring 2026
Samuel Lee, Sitong Liu, Zach Greenman

In [1]:
import pandas as pd
import numpy as np

from mtg_parser.utils.paths import RAW_DIR, PROCESSED_DIR
from mtg_parser.data.extract_creatures import build_creatures_dataset
from mtg_parser.data.build_dataset import save_cleaned_dataset

%load_ext autoreload
%autoreload 2

Preprocessing stuff for our dataset

In [2]:
# Build creatures dataset from default-cards
build_creatures_dataset()

# Save the cleaned dataset
save_cleaned_dataset()

[pipeline] Default-cards dataset already exists.
[pipeline] Creatures dataset already exists.
[cleaning] Missing oracle_id rows: 0
[cleaning] Duplicate values in rows: 0
[builder] Saved combined_points_cleaned.csv to /data/processed/


In [3]:
from mtg_parser.utils.paths import PROCESSED_DIR

cards_df = pd.read_csv(PROCESSED_DIR / 'combined_points_cleaned.csv')

cards_df

,oracle_id,name,set,set_name,released_at,year,cmc,power,toughness,power_numeric,toughness_numeric,keywords,oracle_text,color_identity,rarity,points
0,4c81cfb7-8765-4e28-ae33-4287fa9a86cc,Benalish Hero,lea,Limited Edition Alpha,1993-08-05,1993,1.0,1,1,1.0,1.0,['Banding'],"Banding (Any creatures with banding, and up to...",['W'],common,3
1,218d9277-c179-4de3-9c7f-79b5a6d4fa38,Merfolk of the Pearl Trident,lea,Limited Edition Alpha,1993-08-05,1993,1.0,1,1,1.0,1.0,[],NaN,['U'],common,2
2,10bc98b0-3fdc-46d1-8d3b-6d160e9dd62f,Goblin Balloon Brigade,lea,Limited Edition Alpha,1993-08-05,1993,1.0,1,1,1.0,1.0,[],{R}: This creature gains flying until end of t...,['R'],uncommon,3
3,d3a0b660-358c-41bd-9cd2-41fbf3491b1a,Birds of Paradise,lea,Limited Edition Alpha,1993-08-05,1993,1.0,0,1,0.0,1.0,['Flying'],Flying\n{T}: Add one mana of any color.,['G'],rare,4
4,8b60fcfe-fb90-4a00-a708-25b59bfc9b5a,Will-o'-the-Wisp,lea,Limited Edition Alpha,1993-08-05,1993,1.0,0,1,0.0,1.0,['Flying'],Flying (This creature can't be blocked except ...,['B'],rare,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852,2c75623c-59f4-4449-ab43-9d1225185ad9,Rottenmouth Viper,blb,Bloomburrow,2024-08-02,2024,6.0,6,6,6.0,6.0,[],"As an additional cost to cast this spell, you ...",['B'],mythic,16
853,29423a9a-31a6-4605-8b49-a75895038a3a,Wick's Patrol,blb,Bloomburrow,2024-08-02,2024,6.0,5,3,5.0,3.0,['Mill'],"When this creature enters, mill three cards. W...",['B'],uncommon,12
854,e9c6e4e6-0c3f-4048-93e6-161a1221ea36,"Byrke, Long Ear of the Law",blb,Bloomburrow,2024-08-02,2024,6.0,4,4,4.0,4.0,"['Vigilance', 'Double']","Vigilance\nWhen Byrke enters, put a +1/+1 coun...","['G', 'W']",mythic,13
855,3c345e24-1726-45ac-91ca-e348d1e7acfe,Eddymurk Crab,blb,Bloomburrow,2024-08-02,2024,7.0,5,5,5.0,5.0,['Flash'],Flash\nThis spell costs {1} less to cast for e...,['U'],uncommon,15


In [4]:
import mtg_parser.parsing.keyword_router as kr

test_df = cards_df.sample(n=10, random_state=42)

test_df = kr.parse_keywords_column(test_df)

test_df
print(type(test_df['keywords']))

<class 'pandas.core.series.Series'>


In [5]:
import mtg_parser.pipeline.orchestrator as orch

bundles = orch.extract_abilities(test_df)

bundles

[CardAbilityBundle(oracle_id='33f4fcd5-4eea-4146-a564-da5511a8ee0a', name='Thought Shucker', set='blb', set_name='Bloomburrow', released_at='2024-08-02', cmc=2.0, abilities=[Ability(type=<AbilityType.ACTIVATED: 'activated'>, raw='Threshold — {1}{U}: Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', effect='Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', condition=None, cost='Threshold — {1}{U}')]),
 CardAbilityBundle(oracle_id='b2b76070-f103-4f83-ad21-119e6352f2cd', name='Glowstone Recluse', set='iko', set_name='Ikoria: Lair of Behemoths', released_at='2020-04-24', cmc=3.0, abilities=[Ability(type=<AbilityType.TRIGGERED: 'triggered'>, raw='Whenever this creature mutates, put two +1/+1 counters on it.', effect='put two +1/+1 counters on it.', condition='Whenever this creature mutates', cost=None)]),
 CardAbilityBund

In [6]:
for bundle in bundles:
    print(bundle.abilities)

[Ability(type=<AbilityType.ACTIVATED: 'activated'>, raw='Threshold — {1}{U}: Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', effect='Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', condition=None, cost='Threshold — {1}{U}')]
[Ability(type=<AbilityType.TRIGGERED: 'triggered'>, raw='Whenever this creature mutates, put two +1/+1 counters on it.', effect='put two +1/+1 counters on it.', condition='Whenever this creature mutates', cost=None)]
[Ability(type=<AbilityType.ACTIVATED: 'activated'>, raw='{2}{R}, {T}: Destroy target nonbasic land.', effect='Destroy target nonbasic land.', condition=None, cost='{2}{R}, {T}')]
[Ability(type=<AbilityType.TRIGGERED: 'triggered'>, raw='Whenever this creature deals combat damage to a player, you may draw a card.', effect='you may draw a card.', condition='Whenever this creature de

In [7]:
import mtg_parser.features.keyword_features as kf

keyword_features = [kf.build_keyword_features(row) for _, row in test_df.iterrows()]

keyword_features

RAW KEYWORDS: ['Threshold']
RAW KEYWORDS: ['Reach', 'Mutate']
RAW KEYWORDS: ['Morph']
RAW KEYWORDS: ['Morph', 'Trample']
RAW KEYWORDS: []
RAW KEYWORDS: ['Vigilance']
RAW KEYWORDS: ['Forage', 'Offspring']
RAW KEYWORDS: []
RAW KEYWORDS: ['Cycling']
RAW KEYWORDS: ['Flying']


[KeywordFeatures(features={}),
 KeywordFeatures(features={'reach': 1, 'mutate': 1}),
 KeywordFeatures(features={'morph': 1}),
 KeywordFeatures(features={'morph': 1, 'trample': 1}),
 KeywordFeatures(features={}),
 KeywordFeatures(features={'vigilance': 1}),
 KeywordFeatures(features={'forage': 1, 'offspring': 1}),
 KeywordFeatures(features={}),
 KeywordFeatures(features={'cycling': 1}),
 KeywordFeatures(features={'flying': 1})]

In [8]:
import mtg_parser.features.card_features as cf

cards = []

for _, row in test_df.iterrows():
    card = cf.build_card_features(orch.build_card_ability_bundle(row), row)
    cards.append(card)

print(cards[0])

RAW KEYWORDS: ['Threshold']
RAW KEYWORDS: ['Reach', 'Mutate']
RAW KEYWORDS: ['Morph']
RAW KEYWORDS: ['Morph', 'Trample']
RAW KEYWORDS: []
RAW KEYWORDS: ['Vigilance']
RAW KEYWORDS: ['Forage', 'Offspring']
RAW KEYWORDS: []
RAW KEYWORDS: ['Cycling']
RAW KEYWORDS: ['Flying']
CardFeatures(abilities=[Ability(type=<AbilityType.ACTIVATED: 'activated'>, raw='Threshold — {1}{U}: Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', effect='Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once.', condition=None, cost='Threshold — {1}{U}')], keyword_features=KeywordFeatures(features={}), power=1.0, toughness=3.0)


In [9]:
import mtg_parser.scoring.ability_scorer as a_scorer

ability_scores = []

for card in cards:
    ability_score = a_scorer.get_ability_scores(card.abilities)
    ability_scores.append(ability_score)

for i, ability_score_list in enumerate(ability_scores):
    print(f"\nCARD {i}")

    for a in ability_score_list:
        print(a.ability.raw, "->", a.score)

    print("TOTAL:", sum(a.score for a in ability_score_list))


CARD 0
Threshold — {1}{U}: Put a +1/+1 counter on this creature and draw a card. Activate only if there are seven or more cards in your graveyard and only once. -> 3
TOTAL: 3

CARD 1
Whenever this creature mutates, put two +1/+1 counters on it. -> 2
TOTAL: 2

CARD 2
{2}{R}, {T}: Destroy target nonbasic land. -> 2
TOTAL: 2

CARD 3
Whenever this creature deals combat damage to a player, you may draw a card. -> 2
TOTAL: 2

CARD 4
Whenever this creature becomes tapped, you may pay {1}{U}. If you do, return target permanent to its owner's hand. -> 2
TOTAL: 2

CARD 5
Whenever Clement or another creature you control enters, return up to one target creature you control with lesser mana value to its owner's hand. -> 1
Frogs you control have "{T}: Add {G} or {U}. Spend this mana only to cast a creature spell." -> 3
TOTAL: 4

CARD 6
When this creature enters, you may forage. If you do, put two +1/+1 counters on it. -> 2
TOTAL: 2

CARD 7
As an additional cost to cast this spell, reveal an Element

In [10]:
from mtg_parser.constants.features import ABILITY_PIPELINE

ability = cards[5].abilities[1]
print(ability)
features = ABILITY_PIPELINE.transform_one(ability)

print('features:', features)

Ability(type=<AbilityType.STATIC: 'static'>, raw='Frogs you control have "{T}: Add {G} or {U}. Spend this mana only to cast a creature spell."', effect='Frogs you control have "{T}: Add {G} or {U}. Spend this mana only to cast a creature spell."', condition=None, cost=None)
features: AbilityFeatures(baseline_score=1, has_search=0, plus1_counter=0, enters_trigger=0, dies_trigger=0, repeatable_trigger=0, token_pt_total=0, activated_limiter=0, activated_no_limiter=0, has_global=1, card_advantage=0, has_recursion_battlefield=0, has_exile_access=0, has_destroy=0, has_mass_destroy=0, has_bounce=0, has_deck_stack=0, has_minus_X=0, has_mass_minus_X=0, mana_produced=1, mana_reduction=0, has_direct_damage=0)


In [15]:
import mtg_parser.scoring.keyword_scorer as k_scorer

keyword_scores = []

for card in cards:
    keyword_score = k_scorer.score_keywords(card.keyword_features)
    keyword_scores.append(keyword_score)

for card, score in zip(cards, keyword_scores):
    print('Keywords', card.keyword_features.features)
    print('TOTAL:', score)

Keywords {}
TOTAL: 0
Keywords {'reach': 1, 'mutate': 1}
TOTAL: 2
Keywords {'morph': 1}
TOTAL: 1
Keywords {'morph': 1, 'trample': 1}
TOTAL: 2
Keywords {}
TOTAL: 0
Keywords {'vigilance': 1}
TOTAL: 1
Keywords {'forage': 1, 'offspring': 1}
TOTAL: 4
Keywords {}
TOTAL: 0
Keywords {'cycling': 1}
TOTAL: 1
Keywords {'flying': 1}
TOTAL: 1


In [16]:
import mtg_parser.scoring.pt_scorer as pt_scorer

pt_scores = []

for card in cards:
    pt_score = pt_scorer.score_pt(card.power, card.toughness)
    pt_scores.append(pt_score)

for card, score in zip(cards, pt_scores):
    print('Power', card.power)
    print('Toughness', card.toughness)
    print('TOTAL:', score)

Power 1.0
Toughness 3.0
TOTAL: 4.0
Power 2.0
Toughness 3.0
TOTAL: 5.0
Power 1.0
Toughness 1.0
TOTAL: 2.0
Power 3.0
Toughness 4.0
TOTAL: 7.0
Power 2.0
Toughness 2.0
TOTAL: 4.0
Power 3.0
Toughness 3.0
TOTAL: 6.0
Power 2.0
Toughness 1.0
TOTAL: 3.0
Power 2.0
Toughness 1.0
TOTAL: 3.0
Power 7.0
Toughness 7.0
TOTAL: 17.0
Power 3.0
Toughness 3.0
TOTAL: 6.0


In [21]:
import mtg_parser.scoring.card_scorer as card_scorer

card_scores = []

for card in cards:
    card_score = card_scorer.score_card(card)

    card_scores.append({
        'ability': card_score.ability,
        'keyword': card_score.keyword,
        'pt': card_score.pt,
        'total': card_score.total
    })


for card_score in card_scores:
    print(card_score)

{'ability': 3, 'keyword': 0, 'pt': 4.0, 'total': 7.0}
{'ability': 2, 'keyword': 2, 'pt': 5.0, 'total': 9.0}
{'ability': 2, 'keyword': 1, 'pt': 2.0, 'total': 5.0}
{'ability': 2, 'keyword': 2, 'pt': 7.0, 'total': 11.0}
{'ability': 2, 'keyword': 0, 'pt': 4.0, 'total': 6.0}
{'ability': 4, 'keyword': 1, 'pt': 6.0, 'total': 11.0}
{'ability': 2, 'keyword': 4, 'pt': 3.0, 'total': 9.0}
{'ability': 0, 'keyword': 0, 'pt': 3.0, 'total': 3.0}
{'ability': 1, 'keyword': 1, 'pt': 17.0, 'total': 19.0}
{'ability': 1, 'keyword': 1, 'pt': 6.0, 'total': 8.0}


In [12]:
# def build_scored_df(records: list[CardRecord], human_scores_df) -> pd.DataFrame:
#     rows = []

#     for r in records:
#         score = score_card(r.features)

#         rows.append({
#             "oracle_id": r.oracle_id,
#             "name": r.name,
#             "set": r.set,
#             "released_at": r.released_at,

#             "ability_score": score.ability,
#             "keyword_score": score.keyword,
#             "pt_score": score.pt,
#             "parser_score": score.total,
#         })

#     df = pd.DataFrame(rows)

#     return df.merge(human_scores_df, on="oracle_id", how="left")